# This notebook shows the formation of the semantic web

## 0. Imports

In [28]:
import pandas as pd
import numpy as np
import networkx as nx
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

## 1. Config

In [29]:
FUNC_PREFIX = "function_"          # columns like function_1 ... function_6
SA_SIM_THRESHOLD = 0.95            # source <-> application threshold
FF_SIM_THRESHOLD = 0.70            # function <-> function threshold
MODEL_NAME = "sentence-transformers/all-mpnet-base-v2"

## 2. Import dataframe

In [30]:
df = pd.read_csv('materials_data_preprocessed.csv')
df

,source_clean,function_1,function_2,function_3,function_4,function_5,function_6,application_clean
0,013 denim,recycle yarn,weave fabric,support innovation,connect community,NaN,NaN,large work of art presented to the dutch royal...
1,100 bacterial dye,produce pigments,create sustainable alternative,reduce water usage,minimize energy consumption,NaN,NaN,microbial colour library for dyeing textiles
2,basalt knitted fabric,reinforce fabric,prevent algal growth,extend residence time,conduct heat poorly,resist electricity,NaN,reinforcement fabric for maritime applications
3,100 biobased flax panel,provide structural support,reduce environmental impact,enable biobased composition,NaN,NaN,NaN,interior wall panels
4,100 rejects waxed printed cotton,recycle textiles,create carpets,reduce waste,innovate design,NaN,NaN,high quality recycled carpets
...,...,...,...,...,...,...,...,...
2805,zero furniture panel,reduce co2 emissions,sequester carbon,provide bending strength,facilitate coating,enable sawing,NaN,furniture components for interior design
2806,zero,reduce joint thickness,provide ventilation,absorb rainwater,slow ageing,prevent water penetration,NaN,joint free brick wall construction
2807,zinc foam,absorb energy,increase strength,create foam,cast lightweight structure,NaN,NaN,energy absorbing structural component
2808,zintek titanium zinc,enhance properties,develop patina,provide durability,offer longevity,reduce maintenance,NaN,architectural facades


## 3. Helpers

In [31]:
def node_id(cat, text):
    # ensure uniqueness across categories with a prefix
    return f"{cat[:1].upper()}::{text}"

def add_or_increment_edge(u, v, **attrs):
    if G.has_edge(u, v):
        # increment count for co-occurrence edges
        if "type" in attrs and attrs["type"] == "cooccurrence":
            G[u][v]["weight"] = G[u][v].get("weight", 0) + attrs.get("weight", 1)
        # track max similarity if present
        if "similarity" in attrs:
            G[u][v]["similarity"] = max(G[u][v].get("similarity", 0.0), attrs["similarity"])
    else:
        G.add_edge(u, v, **attrs)

def sa_similarity(src_text, app_text):
    if (src_text not in s_idx) or (app_text not in a_idx):
        return 0.0
    si, ai = s_idx[src_text], a_idx[app_text]
    # cosine between normalized embeddings is dot product
    return float(np.dot(emb_sources[si], emb_applications[ai]))

## 4. Collect unique strings per category

In [32]:
func_cols = [c for c in df.columns if c.startswith(FUNC_PREFIX)]
sources = sorted(set(x for x in df["source_clean"].fillna("").astype(str) if x))
applications = sorted(set(x for x in df["application_clean"].fillna("").astype(str) if x))
# flatten function columns
functions = sorted(
    set(
        f.strip()
        for col in func_cols
        for f in df[col].fillna("").astype(str)
        if f and f.strip()
    )
)

## 5. SBERT Embeddings for Each Category

In [38]:
model = SentenceTransformer(MODEL_NAME)
emb_sources = model.encode(sources, normalize_embeddings=True) if sources else np.zeros((0, 768))
emb_applications = model.encode(applications, normalize_embeddings=True) if applications else np.zeros((0, 768))
emb_functions = model.encode(functions, normalize_embeddings=True) if functions else np.zeros((0, 768))

# quick lookup dicts
s_idx = {t: i for i, t in enumerate(sources)}
a_idx = {t: i for i, t in enumerate(applications)}
f_idx = {t: i for i, t in enumerate(functions)}

# Precompute function-function similarity matrix once
FF_sim = cosine_similarity(emb_functions) if len(functions) > 1 else np.zeros((len(functions), len(functions)))

## 6. Create Graph and add nodes

In [76]:
G = nx.Graph()
# Add nodes with attributes
for t in sources:
    G.add_node(node_id("source", t), label=t, category="source")
for t in applications:
    G.add_node(node_id("application", t), label=t, category="application")
for t in functions:
    G.add_node(node_id("function", t), label=t, category="function")

## 7. Add cooccurrence edges

In [78]:
for _, row in df.iterrows():
    s = str(row.get("source_clean", "") or "").strip()
    a = str(row.get("application_clean", "") or "").strip()
    flist = [str(row.get(c, "") or "").strip() for c in func_cols]
    flist = [f for f in flist if f!='nan']  # non-empty only

    s_id = node_id("source", s) if s else None
    a_id = node_id("application", a) if a else None
    f_ids = [node_id("function", f) for f in flist]

    # source <-> function (co-occurrence)
    if s_id:
        for fid in f_ids:
            add_or_increment_edge(s_id, fid, type="cooccurrence", weight=1)

    # function <-> application (co-occurrence)
    if a_id:
        for fid in f_ids:
            add_or_increment_edge(fid, a_id, type="cooccurrence", weight=1)

## 8. Add function-function similarity edges

In [83]:
nF = len(functions)
for i in range(nF):
    for j in range(i + 1, nF):
        sim = FF_sim[i, j]
        if sim >= FF_SIM_THRESHOLD:
            u = node_id("function", functions[i])
            v = node_id("function", functions[j])
            add_or_increment_edge(u, v, type="similarity", weight=float(sim), similarity=float(sim))

## 9. Save

In [85]:
nx.write_gexf(G, "semantic_web.gexf")
nx.write_graphml(G, "semantic_web.graphml")
print(G.number_of_nodes(), "nodes;", G.number_of_edges(), "edges")


12433 nodes; 45158 edges
